# Módulo 4 — Claude API: Análise Estruturada de Dados

Neste notebook usamos a **API da Anthropic** diretamente via SDK Python para análises que retornam dados estruturados (JSON), ideal para integrar IA em pipelines de dados.

## O que vamos construir

1. Conexão com a API usando `anthropic` SDK
2. Análise de um registro de consumidor e retorno em JSON
3. Processamento em lote: analisar múltiplos registros
4. Geração de relatório narrativo de qualidade
5. Comparação de abordagens: raw vs. structured output

> **Pré-requisito:** `ANTHROPIC_API_KEY` configurada no arquivo `.env`.

In [ ]:
import os
import json
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))

try:
    import anthropic
    print(f"anthropic SDK: OK")
except ImportError:
    print("Instale com: uv add anthropic")
    sys.exit(1)

import pandas as pd

# Base de dados
possible = [
    "../../modulo2_dados_com_pandas/dados/consumidores_simulado.csv",
    "../modulo2_dados_com_pandas/dados/consumidores_simulado.csv",
]
csv_path = next((p for p in possible if Path(p).exists()), None)
df = pd.read_csv(csv_path) if csv_path else pd.DataFrame()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

print(f"Modelo: {MODEL}")
print(f"Base: {len(df)} registros" if not df.empty else "Base não encontrada")

## 1. Chamada básica — diagnóstico de um registro

In [ ]:
# Pegar um registro com problemas da base
if not df.empty:
    registro = df.iloc[6].to_dict()  # registro com outlier de consumo
else:
    registro = {
        "cod_consumidor": 7,
        "nom_consumidor": "Consumidor Teste",
        "cpf_cnpj": None,
        "cod_modalidade_tarifaria": "B1",
        "cod_classe_consumo": "RESIDENCIAL",
        "vlr_consumo_medio_kwh": 9999999,
        "flg_ativo": True,
    }

prompt_diagnostico = f"""
Você é um especialista em qualidade de dados do setor elétrico brasileiro.

Analise o seguinte registro de uma unidade consumidora e identifique problemas:

{json.dumps(registro, ensure_ascii=False, indent=2, default=str)}

Retorne um JSON com exatamente esta estrutura:
{{
  "cod_consumidor": <int>,
  "score_qualidade": <0-100>,
  "problemas": [
    {{"campo": "<nome>", "severidade": "CRITICO|ALTO|MEDIO", "descricao": "<texto>"}}
  ],
  "recomendacao": "<ação corretiva principal>"
}}

Responda APENAS com o JSON, sem texto adicional.
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": prompt_diagnostico}],
)

raw = response.content[0].text
print("Resposta bruta do Claude:")
print(raw)
print()

# Parsear o JSON
resultado = json.loads(raw)
print("JSON parseado:")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

## 2. Processamento em lote com structured output

Agora processamos múltiplos registros, coletando as análises em um DataFrame.

In [ ]:
import time

def analisar_registro_claude(registro: dict) -> dict:
    """Envia um registro ao Claude e retorna análise estruturada em JSON."""
    
    prompt = f"""Analise este registro de consumidor de energia elétrica e retorne JSON:

{json.dumps(registro, ensure_ascii=False, default=str)}

JSON de retorno (apenas o JSON, nada mais):
{{
  "cod_consumidor": <int>,
  "score": <0-100>,
  "status": "OK|ATENCAO|CRITICO",
  "problemas": ["<problema1>", "<problema2>"],
  "acao_sugerida": "<texto curto>"
}}"""

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}],
        )
        return json.loads(response.content[0].text)
    except Exception as e:
        return {
            "cod_consumidor": registro.get("cod_consumidor"),
            "score": 0,
            "status": "ERRO",
            "problemas": [str(e)],
            "acao_sugerida": "Verificar manualmente",
        }


# Processar os primeiros 5 registros (evitar custos altos em demo)
amostra = df.head(5).to_dict(orient="records") if not df.empty else [registro]

resultados = []
for rec in amostra:
    print(f"Analisando UC {rec.get('cod_consumidor')}...", end=" ")
    resultado = analisar_registro_claude(rec)
    resultados.append(resultado)
    print(f"Score={resultado['score']} | {resultado['status']}")
    time.sleep(0.5)  # rate limit

df_resultados = pd.DataFrame(resultados)
print("\nResultados consolidados:")
print(df_resultados[["cod_consumidor", "score", "status", "acao_sugerida"]])

## 3. Relatório narrativo executivo

Passamos o resumo da análise para o Claude gerar um relatório em linguagem natural, pronto para enviar à liderança.

In [ ]:
resumo_base = {
    "total_registros": len(df_resultados),
    "score_medio": round(df_resultados["score"].mean(), 1),
    "distribuicao_status": df_resultados["status"].value_counts().to_dict(),
    "pior_uc": df_resultados.loc[df_resultados["score"].idxmin(), ["cod_consumidor", "score", "problemas"]].to_dict(),
}

prompt_relatorio = f"""
Você é um analista de dados sênior de uma distribuidora de energia elétrica.

Com base na análise de qualidade abaixo, escreva um relatório executivo em português.
O relatório deve ter: situação geral, principais riscos, e recomendações priorizadas.
Seja direto e use linguagem adequada para gestores (sem jargões técnicos de TI).
Máximo 3 parágrafos.

Dados da análise:
{json.dumps(resumo_base, ensure_ascii=False, indent=2, default=str)}
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": prompt_relatorio}],
)

print("RELATÓRIO EXECUTIVO — QUALIDADE DE DADOS DE CONSUMIDORES")
print("="*60)
print(response.content[0].text)
print("="*60)
print(f"\nTokens usados: input={response.usage.input_tokens} | output={response.usage.output_tokens}")

## 4. Conversa multi-turno — o Claude como analista interativo

A API suporta histórico de mensagens, permitindo um diálogo com o modelo sobre os dados.

In [ ]:
# Simular uma conversa de análise de dados
historico = []

def perguntar(pergunta: str, contexto_df: pd.DataFrame = None) -> str:
    """Envia uma pergunta mantendo o histórico da conversa."""
    
    # Na primeira mensagem, incluir contexto dos dados
    if not historico and contexto_df is not None:
        conteudo = f"""Você é um analista de dados do setor elétrico.
        
Tenho os seguintes dados de qualidade de consumidores:
{contexto_df.to_markdown(index=False)}

{pergunta}"""
    else:
        conteudo = pergunta
    
    historico.append({"role": "user", "content": conteudo})
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=historico,
    )
    
    resposta = response.content[0].text
    historico.append({"role": "assistant", "content": resposta})
    return resposta


# Rodada 1
print("TURNO 1")
print("Pergunta: Qual é o panorama geral da qualidade?")
r1 = perguntar("Qual é o panorama geral da qualidade desses registros?", df_resultados)
print(f"Claude: {r1}\n")

# Rodada 2 (sem repetir o contexto — já está no histórico)
print("TURNO 2")
print("Pergunta: Qual a prioridade de correção?")
r2 = perguntar("Quais devo priorizar para correção e por quê?")
print(f"Claude: {r2}\n")

# Rodada 3
print("TURNO 3")
print("Pergunta: Gere uma query SQL para identificar os piores casos.")
r3 = perguntar("Gere uma query SQL Snowflake para identificar UCs com score abaixo de 75 na tabela CONSUMIDORES_UC.")
print(f"Claude: {r3}")

## 5. Dicas de uso da API em produção

| Situação | Abordagem recomendada |
|----------|----------------------|
| Análise de registro único | `messages.create` com prompt direto |
| Lote de registros | Loop com `time.sleep(0.5)` + tratamento de erro |
| Relatório executivo | Consolidar dados antes de enviar ao modelo |
| Conversa interativa | Manter `historico` de mensagens |
| JSON obrigatório | Pedir explicitamente no prompt + `json.loads()` |
| Custo controlado | `max_tokens` explícito + usar amostras em dev |

> **Atenção com custos:** em produção, processe em lote apenas quando necessário. 
> Para análises recorrentes, considere cachear os resultados no Snowflake.